# 14 · Forecast wind verification & empirical blend weights

Evaluates archived Open-Meteo weather forecasts against the real winds pilots measured in thermals (`models/wind_check/groups.parquet` from notebook 11), and derives empirical weights for blending forecast models in the live task planner.

### Method

1. **Per-model resumable downloads:** Section 4 gives each forecast model its own cell so downloads can be paced across Open-Meteo's free-tier hourly (5,000 weighted calls/hr) and daily (10,000 weighted calls/day) limits. Every model caches hourly wind profiles (`10 m`, `100 m` or `80/120 m`, and `925/850/700/600/500 hPa`) in `database/weather/hist_forecast/` and pauses cleanly if a quota limit is reached.
2. **Exact vertical matching:** each model's wind profile is interpolated to each pooled thermal group's exact altitude (`h_asl`), skipping pressure levels below the model ground, matching notebook 11's ERA5 scoring.
3. **Fair cross-archive scoring:** because forecast archives start in different years, every model is scored alongside ERA5 on the **exact same rows** (`rel_rmse_vs_era5 = RMSE_model / RMSE_ERA5`), and the core global models are additionally compared head-to-head on their exact common intersection.
4. **Leakage-free grouped validation:** weights are fitted on `free air` + `middle` bands (above 400 m AGL or above local crests) using Empirical Bayes shrinkage (`site → region → global`) and validated with **5-fold grouped cross-validation by competition** (`event_key`) and 1,000-iteration event-clustered bootstrap 95% confidence intervals against equal-weight mean, median, and best single model.

Writes `database/models/forecast_wind_weights/` (`matched_groups.parquet`, `model_scores.csv`, `by_site.csv`, `cv_comparison.json`, `forecast_weights.json`) and copies `forecast_weights.json` to `database/web_data/`.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
BRANCH = "main"                                          # branch to pull from GitHub
REPO = "/content/Pre-task-analyzer"

FROM_YEAR = 2022        # earliest competition year to query (Open-Meteo forecast archives start ~2021-2022)
PAUSE_S = 0.4           # delay between Open-Meteo requests (seconds) to avoid burst 429 rate limits
MIN_CHIP_GROUPS = 1     # set to 2 or 3 to skip rare outlier chips with very few groups
MAX_SITES = None        # set to an integer (e.g. 20) for a quick smoke test; None runs every site

Review these settings before mounting Drive:
- Keep `FROM_YEAR = 2022` and `MIN_CHIP_GROUPS = 1` for full coverage across all sites and competitions.
- Because Open-Meteo enforces hourly and daily weighted call limits on the free tier, run cells in Section 4 one model at a time (or a couple of regional models together), then run Section 5 onward whenever you want to score all models cached so far.

## 2. Mount Drive, get the code

In [ ]:
import json, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
CACHE_DIR = os.path.join(DB, "weather", "hist_forecast")
OUT = os.path.join(DB, "models", "forecast_wind_weights")
assert os.path.isfile(os.path.join(DB, "models", "wind_check", "groups.parquet")), (
    "Run notebook 11 first: database/models/wind_check/groups.parquet is missing"
)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code


def download_model(model_id):
    """Download or resume one forecast model into database/weather/hist_forecast/."""
    cmd = [
        f"{REPO}/scripts/forecast_wind_weights.py",
        DB,
        "--from-year", str(FROM_YEAR),
        "--models", model_id,
        "--pause", str(PAUSE_S),
        "--min-chip-groups", str(MIN_CHIP_GROUPS),
        "--download-only",
    ]
    if MAX_SITES is not None:
        cmd += ["--max-sites", str(MAX_SITES)]
    return run(*cmd)

Drive is mounted, `database/models/wind_check/groups.parquet` is verified present, `BRANCH` is up to date, and `download_model(model_id)` is ready. All required libraries (`numpy`, `pandas`, `pyarrow`, `scikit-learn`, `matplotlib`) come pre-installed in Colab.

## 3. Input data & coverage overview

Inspects the pooled IGC wind measurements from notebook 11 (`groups.parquet`) since `FROM_YEAR`: counts by year, height band, missing values, and top countries.

In [ ]:
g_raw = pd.read_parquet(os.path.join(DB, "models", "wind_check", "groups.parquet"))
g_raw["hour"] = pd.to_datetime(g_raw["hour"])
g_sub = g_raw[g_raw["hour"].dt.year >= FROM_YEAR].copy()
sites_tbl = pd.read_parquet(os.path.join(DB, "sites.parquet"), columns=["site_id", "name", "country", "lat", "lon"])
g_sub = g_sub.merge(sites_tbl, on="site_id", how="left")

print(f"Total groups in groups.parquet: {len(g_raw):,} ({g_raw['hour'].dt.year.min()}–{g_raw['hour'].dt.year.max()})")
print(f"Groups since {FROM_YEAR}: {len(g_sub):,} across {g_sub['site_id'].nunique()} sites, "
      f"{g_sub['chip_id'].nunique()} chips, {g_sub['hour'].dt.date.nunique()} UTC days")
print("\nNull check on core columns:")
print(g_sub[["pilot_u", "pilot_v", "model_u", "model_v", "agl", "h_asl", "lat", "lon"]].isna().sum().to_frame("nulls").T)

print("\nGroups by year and height band:")
yr_band = pd.crosstab(g_sub["hour"].dt.year, g_sub["band"])
yr_band["total"] = yr_band.sum(axis=1)
display(yr_band)

print("\nTop 10 countries by group count:")
display(g_sub.groupby("country", dropna=False).agg(groups=("pilot_u", "size"), sites=("site_id", "nunique")).sort_values("groups", ascending=False).head(10))

Verify that:
- Core wind and altitude columns (`pilot_u`, `pilot_v`, `model_u`, `model_v`, `agl`, `h_asl`) have zero nulls.
- Both `free air` and `middle` bands have broad multi-year coverage from `FROM_YEAR` onward; these two bands represent the ambient flow above terrain channeling and are used to fit and cross-validate the blend weights.

## 4. Download historical forecasts (one cell per model)

Run the cache status check below at any time to see which models are already downloaded on Drive, then run the individual model cells manually (pacing ~1 global model per hour, or several smaller regional models together). Already-fetched `(chip, date)` intervals are automatically skipped on resume, and every model checkpoints to `database/weather/hist_forecast/` every 20 batches and whenever Open-Meteo's hourly/daily quota pauses a run.

In [ ]:
ALL_MODELS = [
    ("ecmwf_ifs", "global"),
    ("gfs", "global"),
    ("icon", "global"),
    ("gem", "global"),
    ("ecmwf_aifs", "global"),
    ("icon_eu", "regional"),
    ("icon_d2", "regional"),
    ("arome", "regional"),
    ("hrdps", "regional"),
    ("rrfs", "regional"),
]
status_rows = []
for mid, kind in ALL_MODELS:
    pq_p = os.path.join(CACHE_DIR, f"{mid}.parquet")
    done_p = os.path.join(CACHE_DIR, f"{mid}_fetched.json")
    n_rows = len(pd.read_parquet(pq_p, columns=["chip_id"])) if os.path.isfile(pq_p) else 0
    n_done = len(json.loads(open(done_p, encoding="utf-8").read())) if os.path.isfile(done_p) else 0
    status_rows.append({"model": mid, "scope": kind, "cached_chip_hours": n_rows, "fetched_keys": n_done})
display(pd.DataFrame(status_rows))

Use the table above to see your current progress on Drive before triggering the next model cell below.

### 4.1 Global · ECMWF IFS (`ecmwf_ifs025`)

In [ ]:
download_model("ecmwf_ifs")

If `ecmwf_ifs` prints `cache complete` (or `(done)`), move on to the next model cell; if it prints `[PAUSED]`, wait for the hourly/daily quota window to reset and re-run this cell.

### 4.2 Global · GFS (`gfs_global`)

In [ ]:
download_model("gfs")

Check whether `gfs` completed or paused cleanly at quota; already-downloaded batches are preserved in `database/weather/hist_forecast/gfs.parquet`.

### 4.3 Global · ICON (`icon_global`)

In [ ]:
download_model("icon")

Check whether `icon` completed or paused cleanly at quota before continuing to `gem`.

### 4.4 Global · GEM (`gem_global`)

In [ ]:
download_model("gem")

Check whether `gem` completed (`gem_global` synthesizes 100 m wind as the vector midpoint of 80 m and 120 m). Once the 4 core globals (`ecmwf_ifs`, `gfs`, `icon`, `gem`) are done, you can already run Section 5 onward for an initial CV benchmark.

### 4.5 Global · ECMWF AIFS (`ecmwf_aifs025_single`)

In [ ]:
download_model("ecmwf_aifs")

Note: ECMWF AIFS entered operational archiving later than the core globals (~2024–2025), so fewer chip-hours are expected; paired `rel_rmse_vs_era5` normalizes for that date range automatically.

### 4.6 Regional (Europe) · ICON-EU (`icon_eu`)

In [ ]:
download_model("icon_eu")

Covers European competition sites (`30°–70° N, 23° W–44.5° E`, 7 km resolution).

### 4.7 Regional (Central Europe / Alps) · ICON-D2 (`icon_d2`)

In [ ]:
download_model("icon_d2")

Covers Central Europe and the Alps (`43.5°–57.5° N, 3.5° W–20° E`, 2 km resolution).

### 4.8 Regional (France / Western Alps) · AROME (`meteofrance_arome_france`)

In [ ]:
download_model("arome")

Covers France and adjacent Alpine/Pyrenean sites (`38°–55° N, 11.5° W–15.5° E`, 1.3 km resolution). Edge chips slightly outside AROME's rotated grid are automatically retried chip-by-chip.

### 4.9 Regional (North America) · HRDPS (`gem_hrdps_continental`)

In [ ]:
download_model("hrdps")

Covers North America (`36°–68° N, 148°–48° W`, 2.5 km resolution; uses 80 m / 120 m midpoint for 100 m wind).

### 4.10 Regional (CONUS) · RRFS (`ncep_rrfs_conus`)

In [ ]:
download_model("rrfs")

Covers CONUS (`22°–52° N, 133°–62° W`, 3 km resolution). Re-run the cache status cell at the top of Section 4 if you want to double-check all 10 models before scoring.

## 5. Score all cached models & run 5-fold grouped CV

Runs `scripts/forecast_wind_weights.py --skip-download` offline against every model cached in `database/weather/hist_forecast/`. Vertically interpolates each model, prints paired scores vs ERA5, evaluates the common global intersection with 95% event-cluster bootstrap CIs, runs 5-fold competition-grouped CV, and exports `forecast_weights.json`.

In [ ]:
cmd = [
    f"{REPO}/scripts/forecast_wind_weights.py",
    DB,
    "--from-year", str(FROM_YEAR),
    "--models", "all",
    "--min-chip-groups", str(MIN_CHIP_GROUPS),
    "--skip-download",
]
if MAX_SITES is not None:
    cmd += ["--max-sites", str(MAX_SITES)]

run(*cmd)

Inspect the summary tables printed above:
- Confirm that every model you downloaded appears in the `free+middle` accuracy table with its expected `first_date` and `groups` count.
- Check `rel_rmse_vs_era5` (values closer to `1.0` match ERA5 reanalysis more closely on the exact same groups) and the 5-fold CV `delta_vs_equal` values.

## 6. Model accuracy comparison & height-band breakdown

Compares every cached forecast model's paired relative RMSE against ERA5 on `free+middle` bands, and shows how vector RMSE varies from `near the ground` (< 400 m AGL, terrain-dominated) up through `middle` and `free air`.

In [ ]:
scores = pd.read_csv(os.path.join(OUT, "model_scores.csv"))
cv = json.loads(open(os.path.join(OUT, "cv_comparison.json"), encoding="utf-8").read())
weights = json.loads(open(os.path.join(OUT, "forecast_weights.json"), encoding="utf-8").read())

fm = scores[scores["band"] == "free+middle"].sort_values("rel_rmse_vs_era5").reset_index(drop=True)
display_cols = [
    "model", "name", "region", "groups", "events", "first_date",
    "vector_rmse_ms", "era5_rmse_ms", "rel_rmse_vs_era5",
    "vector_median_ms", "speed_bias_ms", "dir_error_deg", "dir_within_30",
]
display(fm[display_cols].round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Panel A: Paired relative RMSE vs ERA5 (free+middle)
ax = axes[0]
fm_plot = fm.sort_values("rel_rmse_vs_era5", ascending=False)
colors = ["#2a78d6" if r == "global" else "#2c9f74" for r in fm_plot["region"]]
bars = ax.barh(fm_plot["name"], fm_plot["rel_rmse_vs_era5"], color=colors, alpha=0.88)
ax.axvline(1.0, color="#666666", linestyle="--", lw=1.2, label="ERA5 parity (1.00)")
for bar, (_, row) in zip(bars, fm_plot.iterrows()):
    w = bar.get_width()
    ax.text(
        w + 0.008, bar.get_y() + bar.get_height() / 2,
        f"{w:.3f}  (n={int(row['groups']):,}, {str(row['first_date'])[:4]}+)",
        va="center", fontsize=8.5,
    )
ax.set_xlim(0.8, max(1.25, float(fm_plot["rel_rmse_vs_era5"].max()) + 0.22))
ax.set_xlabel("Vector RMSE relative to ERA5 on identical rows (lower is better)")
ax.set_title("Paired wind skill vs ERA5 (free air + middle)")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

# Panel B: Vector RMSE by height band for global models + ERA5
ax = axes[1]
bands_order = ["near the ground", "middle", "free air"]
glob_models = [m for m in ["ecmwf_ifs", "icon", "gfs", "gem", "ecmwf_aifs"] if m in set(scores["model"])]
x = np.arange(len(bands_order))
n_series = len(glob_models) + 1
width = 0.78 / max(n_series, 1)
palette = ["#2a78d6", "#2c9f74", "#e07b39", "#8c56b8", "#c74d6e"]

for idx, mid in enumerate(glob_models):
    sub = scores[(scores["model"] == mid) & (scores["band"].isin(bands_order))].set_index("band")
    vals = [sub.loc[b, "vector_rmse_ms"] if b in sub.index else np.nan for b in bands_order]
    offset = (idx - (n_series - 1) / 2) * width
    label = sub["name"].iloc[0] if len(sub) else mid
    ax.bar(x + offset, vals, width=width, color=palette[idx % len(palette)], label=label)

if glob_models:
    ref_mid = scores[scores["model"].isin(glob_models)].groupby("model")["groups"].sum().idxmax()
    sub_ref = scores[(scores["model"] == ref_mid) & (scores["band"].isin(bands_order))].set_index("band")
    era_vals = [sub_ref.loc[b, "era5_rmse_ms"] if b in sub_ref.index else np.nan for b in bands_order]
    offset = (len(glob_models) - (n_series - 1) / 2) * width
    ax.bar(x + offset, era_vals, width=width, color="#9e9d93", label="ERA5 (reanalysis)")

ax.set_xticks(x)
ax.set_xticklabels(bands_order)
ax.set_ylabel("Vector RMSE vs pilot thermals (m/s)")
ax.set_title("Global forecast error by height band")
ax.legend(frameon=False, fontsize=8.5, ncol=2)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

How to read these two panels:
- **Left (paired relative RMSE vs ERA5):** blue bars are global models, green bars are regional high-resolution models. Dividing each model by ERA5's RMSE on that model's exact matched dates and chips removes archive-length and geographic bias.
- **Right (height bands):** check how closely each global forecast tracks ERA5 across `near the ground`, `middle`, and `free air`.

## 7. Head-to-head: common global intersection & regional models in-domain

Checks two strict apples-to-apples comparisons on `free+middle`:
1. Core global models (`ecmwf_ifs`, `gfs`, `icon`, `gem`) evaluated on the **exact same groups** with 95% event-clustered bootstrap CIs.
2. Each regional high-resolution model (`icon_d2`, `icon_eu`, `arome`, `hrdps`, `rrfs`) compared directly against the global models available on the **exact same groups inside that regional domain**.

In [ ]:
mg = pd.read_parquet(os.path.join(OUT, "matched_groups.parquet"))
mg_fm = mg[mg["band"].isin(["free air", "middle"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# Panel A: Common intersection of core global models + ERA5 with 95% event-clustered CIs
ax = axes[0]
common = cv.get("common_global") or {}
if common:
    c_items = sorted(common.items(), key=lambda kv: kv[1]["vector_rmse_ms"])
    names = [k for k, _ in c_items]
    rmses = [v["vector_rmse_ms"] for _, v in c_items]
    lo_err = [v["vector_rmse_ms"] - v["rmse_ci95"][0] for _, v in c_items]
    hi_err = [v["rmse_ci95"][1] - v["vector_rmse_ms"] for _, v in c_items]
    bar_colors = ["#9e9d93" if k == "era5" else "#2a78d6" for k in names]
    y = np.arange(len(names))
    ax.barh(y, rmses, xerr=[lo_err, hi_err], color=bar_colors, alpha=0.88, capsize=4, ecolor="#222222")
    for yi, r in zip(y, rmses):
        ax.text(r + max(hi_err) + 0.05, yi, f"{r:.3f} m/s", va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    n_common = next(iter(common.values()))["groups"]
    ax.set_xlabel("Vector RMSE (m/s) with 95% competition-clustered CI")
    ax.set_title(f"Common global intersection ({n_common:,} identical groups)")
    ax.set_xlim(0, max(r + h for r, h in zip(rmses, hi_err)) * 1.18)
else:
    ax.text(0.5, 0.5, "Need >= 2 core global models with overlapping dates", ha="center", va="center")
    ax.set_title("Common global intersection")
ax.spines[["top", "right"]].set_visible(False)

# Panel B: Each regional model vs equal global mean & ERA5 on exact overlapping in-domain groups
ax = axes[1]
regionals = ["icon_d2", "icon_eu", "arome", "hrdps", "rrfs"]
globals_present = [m for m in ["ecmwf_ifs", "gfs", "icon", "gem", "ecmwf_aifs"] if f"{m}_u" in mg_fm.columns]
reg_rows = []
for rmid in regionals:
    if f"{rmid}_u" not in mg_fm.columns:
        continue
    sub = mg_fm[mg_fm[f"{rmid}_u"].notna()].copy()
    if len(sub) < 15:
        continue
    pu, pv = sub["pilot_u"].to_numpy(float), sub["pilot_v"].to_numpy(float)
    r_rmse = float(np.sqrt(np.mean((pu - sub[f"{rmid}_u"]) ** 2 + (pv - sub[f"{rmid}_v"]) ** 2)))
    e_rmse = float(np.sqrt(np.mean((pu - sub["model_u"]) ** 2 + (pv - sub["model_v"]) ** 2)))
    g_u = sub[[f"{m}_u" for m in globals_present]].mean(axis=1).to_numpy(float)
    g_v = sub[[f"{m}_v" for m in globals_present]].mean(axis=1).to_numpy(float)
    ok_g = np.isfinite(g_u) & np.isfinite(g_v)
    g_rmse = float(np.sqrt(np.mean((pu[ok_g] - g_u[ok_g]) ** 2 + (pv[ok_g] - g_v[ok_g]) ** 2))) if ok_g.any() else np.nan
    ifs_rmse = np.nan
    if "ecmwf_ifs_u" in sub.columns and sub["ecmwf_ifs_u"].notna().sum() >= 15:
        ok_i = sub["ecmwf_ifs_u"].notna().to_numpy()
        ifs_rmse = float(np.sqrt(np.mean((pu[ok_i] - sub.loc[ok_i, "ecmwf_ifs_u"]) ** 2 + (pv[ok_i] - sub.loc[ok_i, "ecmwf_ifs_v"]) ** 2)))
    reg_rows.append({
        "regional_model": rmid,
        "groups": int(len(sub)),
        "events": int(sub["event_key"].nunique()),
        "regional_rmse_ms": r_rmse,
        "ecmwf_ifs_rmse_ms": ifs_rmse,
        "global_mean_rmse_ms": g_rmse,
        "era5_rmse_ms": e_rmse,
    })

reg_df = pd.DataFrame(reg_rows)
if len(reg_df):
    x = np.arange(len(reg_df))
    w = 0.25
    ax.bar(x - w, reg_df["regional_rmse_ms"], width=w, color="#2c9f74", label="Regional model")
    ax.bar(x, reg_df["global_mean_rmse_ms"], width=w, color="#2a78d6", label="Global mean (same rows)")
    ax.bar(x + w, reg_df["era5_rmse_ms"], width=w, color="#9e9d93", label="ERA5 (same rows)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r}\n(n={n:,})" for r, n in zip(reg_df["regional_model"], reg_df["groups"])])
    ax.set_ylabel("Vector RMSE (m/s)")
    ax.set_title("Regional high-res models vs global mean in-domain")
    ax.legend(frameon=False, fontsize=8.5)
    display(reg_df.round(3))
else:
    ax.text(0.5, 0.5, "No regional models with >= 15 matched groups", ha="center", va="center")
    ax.set_title("Regional high-res models vs global mean in-domain")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

Key things to examine here:
- **Common global intersection:** exact ranking among core global models on identical competition hours, with 95% event-clustered confidence bars.
- **Regional models in-domain:** verifies whether high-resolution regional models (`icon_d2`, `icon_eu`, `arome`, `hrdps`, `rrfs`) beat the global ensemble inside their footprint.

## 8. Grouped 5-fold CV & learned empirical weights

Evaluates whether learned weights (`global_weighted`, `region_weighted`, `site_weighted`) outperform unweighted baselines (`best_single`, `equal_mean`, `median`) on **unseen competitions** (`GroupKFold` on `event_key`), and inspects the resulting weights exported to `forecast_weights.json`.

In [ ]:
strats = cv.get("strategies") or {}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Panel A: Grouped 5-fold CV comparison
ax = axes[0]
order = ["best_single", "median", "equal_mean", "global_weighted", "region_weighted", "site_weighted", "era5_reanalysis"]
present = [k for k in order if k in strats and strats[k].get("groups", 0) > 0]
if present:
    y = np.arange(len(present))
    rmses = [strats[k]["vector_rmse_ms"] for k in present]
    lo_e, hi_e = [], []
    for k in present:
        ci = strats[k].get("vector_rmse_ci95")
        if ci:
            lo_e.append(strats[k]["vector_rmse_ms"] - ci[0])
            hi_e.append(ci[1] - strats[k]["vector_rmse_ms"])
        else:
            lo_e.append(0.0)
            hi_e.append(0.0)
    colors = [
        "#9e9d93" if k == "era5_reanalysis" else
        "#2c9f74" if "weighted" in k else
        "#e07b39" if k == "best_single" else "#2a78d6"
        for k in present
    ]
    ax.barh(y, rmses, xerr=[lo_e, hi_e], color=colors, alpha=0.88, capsize=4, ecolor="#222222")
    for yi, k, r, h in zip(y, present, rmses, hi_e):
        delta = strats[k].get("delta_rmse_vs_equal_ms")
        d_ci = strats[k].get("delta_rmse_ci95")
        note = f"{r:.3f} m/s"
        if delta is not None and d_ci is not None:
            note += f"  (Δ={delta:+.3f} [{d_ci[0]:+.3f}, {d_ci[1]:+.3f}])"
        ax.text(r + h + 0.04, yi, note, va="center", fontsize=8.5)
    ax.set_yticks(y)
    ax.set_yticklabels(present)
    ax.invert_yaxis()
    ax.set_xlim(0, max(r + h for r, h in zip(rmses, hi_e)) * 1.48)
    ax.set_xlabel("Out-of-fold Vector RMSE (m/s) with 95% CI")
    ax.set_title(f"{cv.get('n_folds', 5)}-fold competition-grouped CV ({cv.get('n_events', 0)} events, β={cv.get('median_beta', '-')})")
else:
    ax.text(0.5, 0.5, str(cv.get("error", "No CV results")), ha="center", va="center")
    ax.set_title("Grouped 5-fold CV")
ax.spines[["top", "right"]].set_visible(False)

# Panel B: Learned global vs regional weights
ax = axes[1]
gw = weights.get("global") or {}
rw = weights.get("by_region") or {}
if gw:
    mids = [k for k, _ in sorted(gw.items(), key=lambda kv: -kv[1])]
    regions_show = [r for r in ["europe", "north_america", "south_america", "rest_of_world"] if r in rw]
    series = [("global", gw)] + [(r, rw[r]) for r in regions_show]
    x = np.arange(len(mids))
    w = 0.8 / max(len(series), 1)
    pal = ["#1f4e79", "#2a78d6", "#2c9f74", "#e07b39", "#8c56b8"]
    for i, (sname, wmap) in enumerate(series):
        vals = [wmap.get(m, 0.0) for m in mids]
        ax.bar(x + (i - (len(series) - 1) / 2) * w, vals, width=w, color=pal[i % len(pal)], label=sname)
    ax.set_xticks(x)
    ax.set_xticklabels(mids, rotation=35, ha="right")
    ax.set_ylabel("Normalized softmax weight")
    ax.set_title("Learned model weights (global & by region)")
    ax.legend(frameon=False, fontsize=8.5)
else:
    ax.text(0.5, 0.5, "No weights exported", ha="center", va="center")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nSite-specific shrunk weights computed for {len(weights.get('by_site', {}))} sites (>= 10 groups & >= 2 competitions).")
by_site_path = os.path.join(OUT, "by_site.csv")
if os.path.isfile(by_site_path):
    bs = pd.read_csv(by_site_path)
    if len(bs):
        piv = bs.pivot_table(index=["site_id", "name", "country"], columns="model", values="vector_rmse_ms")
        tot = bs.groupby(["site_id", "name", "country"])["groups"].max().rename("max_groups")
        top_sites = piv.join(tot).sort_values("max_groups", ascending=False).head(15)
        print("\nTop 15 sites by group count — per-model Vector RMSE (m/s, free+middle):")
        display(top_sites.round(2))

Decision criteria for wiring `forecast_weights.json` into the live API:
- Check whether `global_weighted`, `region_weighted`, or `site_weighted` has a negative `delta_rmse_vs_equal_ms` whose 95% bootstrap CI (`delta_rmse_ci95`) lies below 0 (clear improvement over equal-weight averaging on unseen competitions).
- Prefer the simplest tier (`global`, `by_region`, or `by_site`) that delivers the best out-of-fold RMSE.

## 9. Summary & next steps

### Q&A
- **How accurate are Open-Meteo's operational forecast models against real IGC pilot winds?** Answered by Section 6 (`model_scores.csv`) and Section 7 (`common_global` + in-domain regional comparison), paired against ERA5 reanalysis on identical groups.
- **Does skill-weighted blending beat equal-weight averaging and median on unseen competitions?** Answered by Section 8 (`cv_comparison.json`), using 5-fold competition-grouped cross-validation (`event_key` held out) with 95% event-clustered bootstrap CIs.
- **Where are the empirical weights saved?** `database/models/forecast_wind_weights/forecast_weights.json` (and copied to `database/web_data/forecast_weights.json`).

### Next steps after running all models
1. Record the verified CV scores and model rankings in `docs/results.md`.
2. Wire `database/web_data/forecast_weights.json` into `api/app.py` (which already accepts per-model weights) and expose the empirically validated `Weighted` blend option in the web task planner.